# Single-τ DAS-FedAvg Run (GPU-Friendly)

This notebook runs one τ configuration of `federated_das.py` on a Kaggle/Colab GPU runtime.
It clones your working branch, installs dependencies, runs the validation-selected DAS-FedAvg experiment, and packages outputs for download.

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/Abishek-Chakravarthy/fed-rag.git"
REPO_BRANCH = "das-fedrag"

TAU = 0.0
SEED = 42
ROUNDS = 4
LOCAL_EPOCHS = 1
TARGET = "nfcorpus"

WORKSPACE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO_DIR = WORKSPACE / "fed-rag"
EXP_DIR = REPO_DIR / "zz_coderuns" / "das_fedrag"
EXPORT_DIR = WORKSPACE / f"das_fedavg_exports_{TARGET.replace('-', '_')}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE: {WORKSPACE}")
print(f"REPO_DIR : {REPO_DIR}")
print(f"EXP_DIR  : {EXP_DIR}")

WORKSPACE: /kaggle/working
REPO_DIR : /kaggle/working/fed-rag
EXP_DIR  : /kaggle/working/fed-rag/zz_coderuns/das_fedrag


In [2]:
import os
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git", "clone", "--depth", "1",
        "--branch", REPO_BRANCH, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ],
    check=True,
)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "accelerate", "datasets<3.0.0", "flwr", "pyarrow", "pydantic",
    "pydantic-settings", "transformers==4.48.0", "sentence-transformers==3.4.1",
    "peft", "matplotlib", "pandas", "tqdm",
], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Clone + install complete")

Cloning into '/kaggle/working/fed-rag'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.6.1 which is incompatible.
black 26.3.1 requires pathspec>=1.0.0, but you have pathspec 0.12.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.6 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 46.0.6 which is incom

Clone + install complete


In [3]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("mps available :", torch.backends.mps.is_available())

if torch.cuda.is_available():
    print("gpu device:", torch.cuda.get_device_name(0))
    import subprocess
    subprocess.run(["nvidia-smi"])
elif torch.backends.mps.is_available():
    print("Using Apple Metal backend")
else:
    print("WARNING: No GPU backend detected.")

torch version: 2.10.0+cu128
cuda available: True
mps available : False
gpu device: Tesla T4
Thu Mar 26 11:30:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |     

In [4]:
def build_run_slug(tau, seed, rounds, local_epochs, target):
    target_slug = target.replace("-", "_")
    return f"tau_{tau:.2f}_seed_{seed}_target_{target_slug}_r{rounds}_e{local_epochs}"

RUN_SLUG = build_run_slug(TAU, SEED, ROUNDS, LOCAL_EPOCHS, TARGET)
print(RUN_SLUG)

tau_0.00_seed_42_target_nfcorpus_r4_e1


In [5]:
import re
import subprocess
import sys
from tqdm.auto import tqdm

cmd = [
    sys.executable, "-u", "federated_das.py",
    "--tau", str(TAU),
    "--seed", str(SEED),
    "--rounds", str(ROUNDS),
    "--local-epochs", str(LOCAL_EPOCHS),
    "--target", TARGET,
]

print("Running:", " ".join(cmd))
round_pattern = re.compile(r"Round\s+(\d+)\s+\|")
progress = tqdm(total=ROUNDS, desc=f"target={TARGET} tau={TAU:.2f}", unit="round")

process = subprocess.Popen(
    cmd, cwd=EXP_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

for line in process.stdout:
    print(line, end="")
    match = round_pattern.search(line)
    if match:
        progress.n = min(int(match.group(1)), ROUNDS)
        progress.refresh()

process.wait()
if process.returncode == 0:
    progress.n = ROUNDS
    progress.refresh()
progress.close()

if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd)

Running: /usr/bin/python3 -u federated_das.py --tau 0.0 --seed 42 --rounds 4 --local-epochs 1 --target nfcorpus


target=nfcorpus tau=0.00:   0%|          | 0/4 [00:00<?, ?round/s]

2026-03-26 11:30:34.028176: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774524634.251924     142 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774524634.316420     142 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774524634.839198     142 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774524634.839243     142 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774524634.839246     142 computation_placer.cc:177] computation placer alr

In [6]:
import json
import pandas as pd
import shutil
import zipfile

csv_path = EXP_DIR / "output_csv_files" / f"results_{RUN_SLUG}.csv"
manifest_path = EXP_DIR / "output_csv_files" / f"manifest_{RUN_SLUG}.json"
acceptance_path = EXP_DIR / "output_csv_files" / f"acceptance_{RUN_SLUG}.json"
log_path = EXP_DIR / "output_log_files" / f"log_{RUN_SLUG}.log"

for path in [csv_path, manifest_path, acceptance_path, log_path]:
    print(path.name, path.exists())

run_export_dir = EXPORT_DIR / RUN_SLUG
run_export_dir.mkdir(parents=True, exist_ok=True)

for path in [csv_path, manifest_path, acceptance_path, log_path]:
    if path.exists():
        shutil.copy2(path, run_export_dir / path.name)

zip_path = EXPORT_DIR / f"{RUN_SLUG}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in run_export_dir.iterdir():
        zf.write(file_path, arcname=file_path.name)

print(f"\nExport folder: {run_export_dir}")
print(f"Zip file     : {zip_path}")

df = pd.read_csv(csv_path)
display(df)

with open(acceptance_path, "r", encoding="utf-8") as f:
    acceptance = json.load(f)
print("\nAcceptance report:")
print(json.dumps(acceptance, indent=2))

results_tau_0.00_seed_42_target_nfcorpus_r4_e1.csv True
manifest_tau_0.00_seed_42_target_nfcorpus_r4_e1.json True
acceptance_tau_0.00_seed_42_target_nfcorpus_r4_e1.json True
log_tau_0.00_seed_42_target_nfcorpus_r4_e1.log True

Export folder: /kaggle/working/das_fedavg_exports_nfcorpus/tau_0.00_seed_42_target_nfcorpus_r4_e1
Zip file     : /kaggle/working/das_fedavg_exports_nfcorpus/tau_0.00_seed_42_target_nfcorpus_r4_e1.zip


,round,tau,seed,target_dataset,avg_loss,aggregated_delta_norm,aggregated_model_hash,num_selected,num_total,best_round_so_far,...,client_3_num_examples,client_3_size_weight,client_3_delta_norm,client_4_domain,client_4_relevance,client_4_selected,client_4_loss,client_4_num_examples,client_4_size_weight,client_4_delta_norm
0,1,0.0,42,nfcorpus,0.001688,0.029068,a961d43812c61ebda943d0c59911c762,1,5,1,...,NaN,NaN,NaN,biomedical,0.3212,0,NaN,NaN,NaN,NaN



Acceptance report:
{
  "distinct_round_trajectories": {
    "applicable": true,
    "evidence": {
      "rounds": 1,
      "unique_hashes": 1
    },
    "passed": true
  },
  "positive_tau_excludes_some_clients": {
    "applicable": false,
    "passed": null
  },
  "relevance_scores_ordered": {
    "applicable": true,
    "evidence": {
      "scores": {
        "0": "1.0000",
        "1": "0.4791",
        "2": "0.0130",
        "3": "0.0422",
        "4": "0.3212"
      }
    },
    "passed": true
  },
  "selection_matches_threshold_policy": {
    "applicable": true,
    "evidence": {
      "per_round": [
        false
      ]
    },
    "passed": false
  },
  "target_client_always_selected": {
    "applicable": true,
    "passed": false
  },
  "tau_zero_selects_all": {
    "applicable": true,
    "passed": false
  }
}


## Download

Suggested first pass:
- run one notebook per τ value, keeping `SEED=42`, `ROUNDS=4`, `LOCAL_EPOCHS=1`, and `TARGET="nfcorpus"`
- collect the zipped outputs for each τ and compile them locally

Download either:
- the `.zip` file shown above, or
- the whole `das_fedavg_exports/<run_slug>/` folder

After collecting all τ runs, use `compile_tau_results.py` locally to generate summary plots.